# 环境和分布偏移核心知识点总结

## 一、分布偏移的核心定义

训练数据分布  $p_S(\mathbf{x},y)$  与测试/部署数据分布  $p_T(\mathbf{x},y)$  不一致，是导致模型部署后性能骤降的关键原因之一；若无对分布关系的假设，无法保证模型泛化能力。

## 二、分布偏移的类型

|偏移类型|核心假设（训练→测试）|适用场景|示例|
|---|---|---|---|
|**协变量偏移**|标签条件分布不变（$P(y|\mathbf{x}) $ 不变），输入分布变化（$ P(\mathbf{x})$ 变化）| $\mathbf{x}$  导致  $y$  的场景|
|**标签偏移**|类别条件分布不变（$P(\mathbf{x}|y) $ 不变），标签边缘分布变化（$ P(y)$ 变化）| $y$  导致  $\mathbf{x}$  的场景|
|**概念偏移**|标签的定义（$P(y|\mathbf{x})$）发生变化|类别语义随时间/空间改变|
## 三、分布偏移的典型示例

1. **医学诊断**：用学生血样做健康对照，与患者样本存在年龄/生活习惯偏移，模型仅学习无关特征；

2. **自动驾驶**：用游戏合成数据训练路沿检测器，模型仅学习合成纹理，真实场景失效；

3. **非平稳分布**：广告模型未更新（不知新设备）、垃圾邮件变种逃避检测、推荐系统季节偏移；

4. **其他场景**：人脸检测器缺乏特写样本、搜索引擎跨地区部署、数据集标签均匀但真实分布不均。

## 四、分布偏移的纠正策略

### 1. 核心：经验风险 vs 实际风险

- **经验风险**：训练数据的平均损失（ $\frac{1}{n}\sum_{i=1}^n l(f(\mathbf{x}_i),y_i)$ ），是对**实际风险**（真实分布的期望损失  $E_{p(\mathbf{x},y)}[l(f(\mathbf{x}),y)]$ ）的近似；

- 分布偏移时，需通过**加权经验风险最小化**修正经验风险，使其实近实际风险。

### 2. 协变量偏移纠正

步骤：

1. 构建二元分类集（训练样本标 $-1$ ，测试未标记样本标 $1$ ）；

2. 用对数几率回归训练分类器得到  $h(\mathbf{x})$ ；

3. 计算样本权重  $\beta_i = \exp(h(\mathbf{x}_i))$ （或截断为常量  $c$ ）；

4. 使用权重  $\beta_i$  训练原模型。

### 3. 标签偏移纠正

步骤：

1. 用训练数据训练初始分类器，计算验证集上的混淆矩阵  $\mathbf{C}$ （ $c_{ij}$ ：真实标签 $j$ 被预测为 $i$ 的比例）；

2. 计算测试集的平均预测  $\mu(\hat{\mathbf{y}})$ ，通过线性系统  $\mathbf{C}p(\mathbf{y})=\mu(\hat{\mathbf{y}})$  估计目标标签分布  $p(y)$ ；

3. 计算样本权重  $\beta_i = \frac{p(y_i)}{q(y_i)}$ （ $q(y)$  为训练标签分布）；

4. 使用权重  $\beta_i$  训练原模型。

### 4. 概念偏移纠正

- 缓慢偏移：用新数据**在线更新模型权重**（而非重新训练）；

- 极端偏移：重新收集标签、从零训练模型。

## 五、学习问题的分类法

|学习类型|核心特点|示例|
|---|---|---|
|批量学习|一次训练所有数据，部署后不更新|智能猫门的猫检测器|
|在线学习|逐样本学习，观测预测后更新模型，循环迭代|股票价格预测|
|老虎机|仅有限个“行动”可选，理论最优性保证更强|有限广告位的点击率优化|
|控制|环境反应依赖历史行为，需构建环境模型|咖啡锅炉温度控制|
|强化学习|基于环境行动以最大化预期利益，环境与模型互动|自动驾驶、围棋AI|
## 六、机器学习的公平、责任与透明度

- **反馈循环风险**：模型决策影响环境，进而改变数据分布（如预测性警务的“高巡逻→多抓捕→更高预测”循环）；

- **伦理考量**：需考虑决策的成本敏感性（如错误分类的社会影响）、亚群体公平性，避免算法歧视；

- **核心原则**：部署模型时需监控实时系统，关注模型与环境的意外纠缠。
> 20260117_2018